In [ ]:
from jetutils.plots import *
from jetutils.definitions import *
from jetutils.data import *
from jetutils.jet_finding import coarsen_da, haversine_from_dl, average_jet_categories, jet_integral_haversine, add_normals, jet_position_as_da, extract_features
from jetutils.anyspell import *
import altair as alt

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline

In [ ]:
labels = xr.open_dataarray(
    "/Users/bandelol/Documents/code_local/data/labels_som_6_6_pbc_euclidean.nc"
)
net = XPySom(6, 6)
labels_df = pl.from_pandas(labels.to_dataframe().reset_index())
ny = labels_df["time"].dt.year().n_unique()
ne = labels_df["member"].n_unique()
labels_first_week = labels_df.filter(pl.col("time").dt.ordinal_day() < 161)
labels_JA = labels_df.filter(pl.col("time").dt.month().is_in([7, 8]))
labels_JA = labels_JA.with_columns(year=pl.col("time").dt.year())
vc = labels_first_week["labels"].value_counts(sort=True)

In [ ]:
first_week_pops = labels_first_week["labels"].value_counts().sort("labels")
first_week_pops = first_week_pops.with_columns(pl.col("count") / ny / ne)
first_week_pops_values = np.zeros(net.x * net.y)
first_week_pops_values[first_week_pops["labels"]] = first_week_pops["count"]
cmap = colormaps.bubblegum_r
coords = net.neighborhoods.coordinates
norm = BoundaryNorm(np.arange(-1.0, 1.2, 0.4), cmap.N)
fig, ax = net.plot_on_map(
    first_week_pops_values,
    smooth_sigma=0,
    show=True,
    cmap=cmap,
    norm=norm,
    discretify=True,
    draw_cbar=True,
)
for i, c in enumerate(coords):
    x, y = c
    color = "white" if first_week_pops_values[i] > 0.6 else "black"
    ax.text(x, y, f"${i + 1}$", va="center", ha="center", color=color, fontsize=14)
JA_pops = labels_JA["labels"].value_counts().sort("labels")
JA_pops = JA_pops.with_columns(pl.col("count") / ny / ne)
JA_pops_values = np.zeros(net.x * net.y)
JA_pops_values[JA_pops["labels"]] = JA_pops["count"]
fig, ax = net.plot_on_map(
    JA_pops_values,
    smooth_sigma=0,
    show=True,
    cmap=cmap,
    discretify=True,
    draw_cbar=True,
)
for i, c in enumerate(coords):
    x, y = c
    color = "white" if JA_pops_values[i] > 1.6 else "black"
    ax.text(x, y, f"${i + 1}$", va="center", ha="center", color=color, fontsize=14)

## when does summer start?

In [ ]:
props_cesm = pl.read_parquet(
    "/Users/bandelol/Documents/code_local/data/props_cesm.parquet"
)

In [ ]:
labels_df = xarray_to_polars(labels)
props_on_som = labels_df.cast({"time": pl.Datetime("ns")}).join(
    props_cesm.cast({"time": pl.Datetime("ns")}), on=["member", "time"]
)
props_on_som = (
    props_on_som.group_by(["labels", "jet"])
    .mean()
    .sort(["labels", "jet"], descending=[False, True])[["labels", "jet", "mean_lat"]]
)

In [ ]:
net.plot_on_map(props_on_som[::2, "mean_lat"].to_numpy(), discretify=True)

In [ ]:
net.plot_on_map(props_on_som[1::2, "mean_lat"].to_numpy(), discretify=True)

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(10, 8), subplot_kw={"aspect": "equal"})
axes = axes.flatten()
for i, (ind, df_) in enumerate(
    labels_df.filter(pl.col("time").dt.ordinal_day() < 168).group_by(
        pl.col("time").dt.day(), maintain_order=True
    )
):
    this_pop = df_["labels"].value_counts().sort("labels")
    this_pop = (
        pl.Series("labels", np.arange(36))
        .to_frame()
        .join(this_pop, how="left", on="labels")
        .fill_null(0)
    )
    if i == 0:
        first_day_pop = this_pop
    pop_numpy = this_pop["count"].to_numpy()
    cmap = colormaps.matter
    norm = BoundaryNorm(np.arange(0, 600, 50), cmap.N)
    fig, axes[i] = net.plot_on_map(pop_numpy, fig=fig, ax=axes[i], cmap=cmap, norm=norm)
    axes[i].set_title(df_[0, "time"])
first_day_clu = first_day_pop.filter(pl.col("count") > 50)["labels"].to_numpy()

In [ ]:
def correlate_methods(args):
    thresh_lat, thresh_time_1, thresh_time_2 = args
    thresh_time_1 = int(thresh_time_1)
    thresh_time_2 = int(thresh_time_2)
    all_year_mean_lats = (
        props_cesm.group_by(
            ["member", pl.col("time").dt.year().alias("year"), "jet"],
            maintain_order=True,
        )
        .agg(pl.col("time").dt.ordinal_day().alias("dayofyear"), pl.col("mean_lat"))
        .explode("dayofyear", "mean_lat")
        .filter(pl.col("jet") == "STJ")
    )
    run_lengths_mean_lat = (
        all_year_mean_lats.group_by(["member", "year"], maintain_order=True)
        .agg((pl.col("mean_lat").fill_null(strategy="forward") > thresh_lat).rle())
        .explode("mean_lat")
        .unnest("mean_lat")
    )
    run_lengths_mean_lat = run_lengths_mean_lat.with_columns(
        run_lengths_mean_lat.group_by(["member", "year"], maintain_order=True)
        .agg(
            start=pl.lit(0).append(
                pl.col("len").cum_sum().slice(0, pl.col("len").len() - 1)
            )
        )
        .explode("start")
    )
    start_summer_mean_lat = (
        run_lengths_mean_lat.filter(pl.col("value"), pl.col("len") > thresh_time_1)
        .group_by(["member", "year"], maintain_order=True)
        .first()
    )

    all_year_labels = (
        labels_df.group_by(
            ["member", pl.col("time").dt.year().alias("year")], maintain_order=True
        )
        .agg(pl.col("time").dt.ordinal_day().alias("dayofyear"), pl.col("labels"))
        .explode("dayofyear", "labels")
    )

    run_lengths_labels = (
        all_year_labels.group_by(["member", "year"], maintain_order=True)
        .agg(
            pl.col("labels").is_in(first_day_clu).not_().rle(),
            pl.col("dayofyear").first(),
        )
        .explode("labels")
        .unnest("labels")
    )
    run_lengths_labels = run_lengths_labels.with_columns(
        run_lengths_labels.group_by(["member", "year"], maintain_order=True)
        .agg(
            start=pl.lit(0).append(
                pl.col("len").cum_sum().slice(0, pl.col("len").len() - 1)
            )
            + pl.col("dayofyear")
        )
        .explode("start")
    )
    start_summer_labels = (
        run_lengths_labels.filter(pl.col("value"), pl.col("len") > thresh_time_2)
        .group_by(["member", "year"], maintain_order=True)
        .first()
    )

    start_summer_labels_both = start_summer_labels.drop(
        "len", "value", "dayofyear"
    ).join(start_summer_mean_lat.drop("len", "value"), on=["member", "year"])

    corr = (
        start_summer_labels_both.group_by("member", maintain_order=True)
        .agg(pl.corr(pl.col("start"), pl.col("start_right")))
        .mean()
    )
    return 1 - corr[0, "start"]

In [ ]:
from scipy.optimize import minimize

x0 = [31, 13, 13]
res = minimize(correlate_methods, x0, method="Nelder-Mead", tol=1e-3)

In [ ]:
all_year_mean_lats = (
    props_cesm.group_by(
        ["member", pl.col("time").dt.year().alias("year"), "jet"], maintain_order=True
    )
    .agg(pl.col("time").dt.ordinal_day().alias("dayofyear"), pl.col("mean_lat"))
    .explode("dayofyear", "mean_lat")
    .filter(pl.col("jet") == "STJ")
)

In [ ]:
run_lengths_mean_lat = (
    all_year_mean_lats.group_by(["member", "year"], maintain_order=True)
    .agg((pl.col("mean_lat").fill_null(strategy="forward") > 3.009e01).rle())
    .explode("mean_lat")
    .unnest("mean_lat")
)
run_lengths_mean_lat = run_lengths_mean_lat.with_columns(
    run_lengths_mean_lat.group_by(["member", "year"], maintain_order=True)
    .agg(
        start=pl.lit(0).append(
            pl.col("len").cum_sum().slice(0, pl.col("len").len() - 1)
        )
    )
    .explode("start")
)
start_summer_mean_lat = (
    run_lengths_mean_lat.filter(pl.col("value"), pl.col("len") > 12)
    .group_by(["member", "year"], maintain_order=True)
    .first()
)

In [ ]:
all_year_labels = (
    labels_df.group_by(
        ["member", pl.col("time").dt.year().alias("year")], maintain_order=True
    )
    .agg(pl.col("time").dt.ordinal_day().alias("dayofyear"), pl.col("labels"))
    .explode("dayofyear", "labels")
)

In [ ]:
run_lengths_labels = (
    all_year_labels.group_by(["member", "year"], maintain_order=True)
    .agg(
        pl.col("labels").is_in(first_day_clu).not_().rle(), pl.col("dayofyear").first()
    )
    .explode("labels")
    .unnest("labels")
)
run_lengths_labels = run_lengths_labels.with_columns(
    run_lengths_labels.group_by(["member", "year"], maintain_order=True)
    .agg(
        start=pl.lit(0).append(
            pl.col("len").cum_sum().slice(0, pl.col("len").len() - 1)
        )
        + pl.col("dayofyear")
    )
    .explode("start")
)
start_summer_labels = (
    run_lengths_labels.filter(pl.col("value"), pl.col("len") > 13)
    .group_by(["member", "year"], maintain_order=True)
    .first()
)

In [ ]:
start_summer_labels_both = start_summer_labels.drop("len", "value", "dayofyear").join(
    start_summer_mean_lat.drop("len", "value"), on=["member", "year"]
)

In [ ]:
start_summer_labels_both.group_by("member", maintain_order=True).agg(
    pl.corr(pl.col("start"), pl.col("start_right"))
).mean()

In [ ]:
plt.plot(
    start_summer_labels.filter(pl.col("member") == pl.col("member").unique().get(4))[
        "start"
    ]
)
plt.plot(
    start_summer_mean_lat.filter(pl.col("member") == pl.col("member").unique().get(4))[
        "start"
    ]
)

In [ ]:
start_summer_mean_lat

In [ ]:
fig, ax = plt.subplots(figsize=(20, 9))
# summer_mean_lats = all_year_mean_lats.filter(pl.col("dayofyear") < 180, pl.col("dayofyear") > 149, pl.col("year") < 2020)
for indexer, traj in all_year_mean_lats.group_by(
    ["member", "year"], maintain_order=True
):
    maybe_red = np.random.choice([False] * 700 + [True])
    plt.plot(
        traj["dayofyear"],
        traj["mean_lat"],
        lw=2 if maybe_red else 0.5,
        color="red" if maybe_red else "black",
        alpha=1.0 if maybe_red else 0.5,
        zorder=100 if maybe_red else 1,
    )

In [ ]:
from typing import Literal


def add_jitter(
    x: np.ndarray,
    intensity: float = 0.1,
    side: Literal["left", "right", "both"] = "both",
):
    rng = np.random.default_rng()
    match side:
        case "left":
            jitter = -rng.random(len(x)) * intensity
        case "right":
            jitter = rng.random(len(x)) * intensity
        case "both":
            jitter = rng.random(len(x)) * intensity * 2 - intensity
    x = x + jitter
    return x

In [ ]:
start_summer_labels.group_by("year", maintain_order=True).mean()["start"]

In [ ]:
plt.scatter(
    add_jitter(start_summer_labels.sort("year", "member")["year"].rle_id().to_numpy()),
    start_summer_labels["start"],
)
plt.plot(
    start_summer_labels.group_by("year", maintain_order=True).mean()["start"],
    color="red",
    zorder=100,
)

## where is first week of june ?

In [ ]:
def to_distrib(
    labels_df: pl.DataFrame, expected_nlabels: int | None = None
) -> pl.DataFrame:
    distrib = (
        labels_df.group_by(["member", "year", "labels"])
        .len()
        .sort(["member", "year", "labels"])
    )
    index_ = distrib[["member", "year"]].unique(maintain_order=True)
    if expected_nlabels is None:
        labels_ = distrib[["labels"]].unique().sort("labels")
    else:
        labels_ = pl.Series("labels", np.arange(expected_nlabels)).to_frame()
    index_ = index_.join(labels_, how="cross")
    distrib = index_.join(
        distrib, on=["member", "year", "labels"], how="left"
    ).fill_null(0)
    return distrib


rng1 = np.random.default_rng()
rng2 = np.random.default_rng()
coords = net.neighborhoods.coordinates

for i in range(4):
    june_winner = vc[i, "labels"]
    filter_ = labels_first_week.group_by(
        ["member", pl.col("time").dt.year()], maintain_order=True
    ).agg(proportion=(pl.col("labels") == june_winner).sum() / pl.col("labels").len())
    filter_ = filter_.filter(pl.col("proportion") > 0.5).rename({"time": "year"})
    when_ = filter_.group_by("year").len().sort("year").with_row_index()
    which_members_ = filter_.group_by("member").len().sort("member").with_row_index()
    ny_ = filter_["year"].n_unique()
    ne_ = filter_["member"].n_unique()
    filtered_JA = filter_.join(labels_JA, on=["member", "year"])

    main_distrib = to_distrib(labels_JA, expected_nlabels=36)
    to_test_distrib = to_distrib(filtered_JA, expected_nlabels=36)
    years_ = labels_JA["year"].unique()
    labels_ = labels_JA["labels"].unique()
    members_ = labels_JA["member"].unique()

    n_draws = 1000
    # draw_from_years = np.repeat(when_["index"], when_["len"] + 10)
    draw_from_years = np.arange(len(years_))
    drawn_years = rng1.choice(draw_from_years, n_draws)
    # draw_from_members = which_members_["index"].to_numpy()
    draw_from_members = np.arange(len(members_))
    drawn_members = rng2.choice(draw_from_members, n_draws)
    indices = (drawn_members * len(years_) + drawn_years) * len(labels_)
    indices = indices[:, None] + labels_.to_numpy()[None, :]
    drawn_distribs = main_distrib["len"].to_numpy()[indices]
    to_test_distrib = (
        to_test_distrib["len"].to_numpy().reshape(filter_.shape[0], len(labels_))
    )
    fig, ax = plt.subplots(figsize=(12, 4))
    markercolor = "orange"
    bp1 = ax.boxplot(
        drawn_distribs,
        positions=np.arange(1, 37) - 0.15,
        widths=0.25,
        showfliers=False,
        showmeans=True,
        patch_artist=True,
        medianprops={"color": markercolor, "linewidth": 3},
        meanprops={"markerfacecolor": markercolor, "markeredgewidth": 0},
    )
    for box in bp1["boxes"]:
        box.set_facecolor("dimgrey")
    bp2 = ax.boxplot(
        to_test_distrib,
        positions=np.arange(1, 37) + 0.15,
        widths=0.25,
        showfliers=False,
        showmeans=True,
        patch_artist=True,
        medianprops={"color": markercolor, "linewidth": 3},
        meanprops={"markerfacecolor": markercolor, "markeredgewidth": 0},
    )
    for box in bp2["boxes"]:
        box.set_facecolor(COLORS[0])
    ticks = ax.set_xticks(np.arange(1, 37), labels=np.arange(1, 37))
    ticklabels = ax.get_xticklabels()
    for i in range(36):
        ttest_res = ttest_ind(
            drawn_distribs[:, i], to_test_distrib[:, i], equal_var=False
        )
        if ttest_res.pvalue < 0.05:
            ticklabels[i].set_color(color=COLORS[1])

    filtered_JA = filtered_JA["labels"].value_counts().sort("labels")
    filtered_JA = filtered_JA.with_columns(pl.col("count") / filter_.shape[0])
    cmap = colormaps.curl
    norm = BoundaryNorm(np.arange(-1.0, 1.2, 0.4), cmap.N)
    filtered_JA_values = np.zeros(net.x * net.y)
    filtered_JA_values[filtered_JA["labels"]] = filtered_JA["count"]
    plt.savefig("/Users/bandelol/Documents/code_local/GM-20240424/figs/distribs")
    fig, ax = net.plot_on_map(
        filtered_JA_values - JA_pops_values,
        smooth_sigma=0,
        show=True,
        cmap=cmap,
        norm=norm,
        draw_cbar=True,
        cbar_label="extra day per summer",
    )
    for i, c in enumerate(coords):
        signif = (
            ttest_ind(
                drawn_distribs[:, i], to_test_distrib[:, i], equal_var=False
            ).pvalue
            < 0.05
        )
        color = COLORS[1] if signif else "black"
        ax.text(
            *c,
            str(i + 1),
            color=color,
            ha="center",
            va="center",
            fontsize=12,
            fontweight="demi",
            usetex=False,
        )
    ax.set_title(f"First week on {june_winner + 1}, {filter_.shape[0]} summers")
    plt.savefig("/Users/bandelol/Documents/code_local/GM-20240424/figs/map")
    break

## where do we jump from 1st week of june ?

In [ ]:
labels_df = xarray_to_polars(labels)
labels_df = labels_df.with_columns(
    year=pl.col("time").dt.year(), dayofyear=pl.col("time").dt.ordinal_day()
)
first_day_clu = labels_df.filter(pl.col("dayofyear") == pl.col("dayofyear").first())[
    "labels"
].value_counts()
first_day_clu = (
    pl.Series("labels", np.arange(36))
    .to_frame()
    .join(first_day_clu, on="labels", how="left")
    .fill_null(0)
)
first_day_clu = first_day_clu.filter(pl.col("count") > 50)["labels"].to_numpy()

In [ ]:
labels_trans = (
    labels_df.filter(pl.col("labels").is_in(first_day_clu).not_())
    .group_by("member", "year", maintain_order=True)
    .head(7)
)
vc = labels_trans["labels"].value_counts().sort("count", descending=True)

In [ ]:
def to_distrib(
    labels_df: pl.DataFrame, expected_nlabels: int | None = None
) -> pl.DataFrame:
    distrib = (
        labels_df.group_by(["member", "year", "labels"])
        .len()
        .sort(["member", "year", "labels"])
    )
    index_ = distrib[["member", "year"]].unique(maintain_order=True)
    if expected_nlabels is None:
        labels_ = distrib[["labels"]].unique().sort("labels")
    else:
        labels_ = pl.Series("labels", np.arange(expected_nlabels)).to_frame()
    index_ = index_.join(labels_, how="cross")
    distrib = index_.join(
        distrib, on=["member", "year", "labels"], how="left"
    ).fill_null(0)
    return distrib


rng1 = np.random.default_rng()
rng2 = np.random.default_rng()
coords = net.neighborhoods.coordinates

for i in range(4):
    june_winner = vc[i, "labels"]
    filter_ = labels_trans.group_by(
        ["member", pl.col("time").dt.year()], maintain_order=True
    ).agg(proportion=(pl.col("labels") == june_winner).sum() / pl.col("labels").len())
    filter_ = filter_.filter(pl.col("proportion") > 0.4).rename({"time": "year"})
    when_ = filter_.group_by("year").len().sort("year").with_row_index()
    which_members_ = filter_.group_by("member").len().sort("member").with_row_index()
    ny_ = filter_["year"].n_unique()
    ne_ = filter_["member"].n_unique()
    filtered_JA = filter_.join(labels_JA, on=["member", "year"])

    main_distrib = to_distrib(labels_JA, expected_nlabels=36)
    to_test_distrib = to_distrib(filtered_JA, expected_nlabels=36)
    years_ = labels_JA["year"].unique()
    labels_ = labels_JA["labels"].unique()

    n_draws = 100000
    draw_from_years = np.repeat(when_["index"], when_["len"])
    drawn_years = rng1.choice(draw_from_years, n_draws)
    draw_from_members = np.repeat(which_members_["index"], which_members_["len"])
    drawn_members = rng2.choice(draw_from_members, n_draws)
    indices = (drawn_members * len(years_) + drawn_years) * len(labels_)
    indices = indices[:, None] + labels_.to_numpy()[None, :]
    drawn_distribs = main_distrib["len"].to_numpy()[indices]
    to_test_distrib = (
        to_test_distrib["len"].to_numpy().reshape(filter_.shape[0], len(labels_))
    )
    with plt.style.context("seaborn-v0_8"):
        fig, ax = plt.subplots(figsize=(12, 4))
        _ = ax.boxplot(
            drawn_distribs,
            positions=np.arange(1, 37) - 0.15,
            widths=0.25,
            showfliers=False,
            showmeans=True,
        )
        _ = ax.boxplot(
            to_test_distrib,
            positions=np.arange(1, 37) + 0.15,
            widths=0.25,
            showfliers=False,
            showmeans=True,
        )

    ticks = ax.set_xticks(np.arange(1, 37), labels=np.arange(1, 37))
    ticklabels = ax.get_xticklabels()
    for i in range(36):
        ttest_res = ttest_ind(
            drawn_distribs[:, i], to_test_distrib[:, i], equal_var=False
        )
        if ttest_res.pvalue < 0.05:
            ticklabels[i].set_color(color="red")

    filtered_JA = filtered_JA["labels"].value_counts().sort("labels")
    filtered_JA = filtered_JA.with_columns(pl.col("count") / filter_.shape[0])
    cmap = colormaps.balance
    norm = BoundaryNorm(np.arange(-1.0, 1.2, 0.4), cmap.N)
    filtered_JA_values = np.zeros(net.x * net.y)
    filtered_JA_values[filtered_JA["labels"]] = filtered_JA["count"]
    fig, ax = net.plot_on_map(
        filtered_JA_values - JA_pops_values,
        smooth_sigma=0,
        show=True,
        cmap=cmap,
        norm=norm,
        draw_cbar=True,
        cbar_label="extra day per summer",
    )
    for i, c in enumerate(coords):
        signif = (
            ttest_ind(
                drawn_distribs[:, i], to_test_distrib[:, i], equal_var=False
            ).pvalue
            < 0.05
        )
        color = COLORS[2] if signif else "black"
        ax.text(
            *c,
            str(i + 1),
            color=color,
            ha="center",
            va="center",
            fontsize=12,
            fontweight="demi",
            usetex=False,
        )
    ax.set_title(f"First week on {june_winner + 1}, {filter_.shape[0]} summers")

## cluster trajectories like weiland?

### cluster jet index trajs

In [ ]:
ts = props_cesm.filter(
    pl.col("time").dt.month().is_in([7, 8]), pl.col("time").dt.year() < 2020
)
varnames = ["mean_lat", "mean_s", "tilt", "width"]
ts_numpy_past = (
    ts[varnames]
    .fill_null(strategy="forward")
    .fill_null(strategy="backward")
    .to_numpy()
    .reshape(49 * 30, -1, len(varnames) * 2)
)

In [ ]:
from tslearn.clustering import TimeSeriesKMeans

model = TimeSeriesKMeans(n_clusters=2, metric="dtw", max_iter=20)
model = model.fit(ts_numpy_past)

In [ ]:
labels_past = model.labels_

In [ ]:
ts = props_cesm.filter(
    pl.col("time").dt.month().is_in([7, 8]), pl.col("time").dt.year() > 2020
)
ts_numpy_future = (
    ts[varnames]
    .fill_null(strategy="forward")
    .fill_null(strategy="backward")
    .to_numpy()
    .reshape(49 * 30, -1, len(varnames) * 2)
)

In [ ]:
model = TimeSeriesKMeans(n_clusters=2, metric="dtw", max_iter=20)
model = model.fit(ts_numpy_future)

In [ ]:
labels_future = model.labels_

In [ ]:
np.unique(labels_future, return_counts=True)

In [ ]:
i = 0

plt.plot(
    np.median(ts_numpy_past[labels_past == 0], axis=0)[:, i], color="black", ls="solid"
)
q1, q2 = np.quantile(ts_numpy_past[labels_past == 0, ..., i], [0.33, 0.66], axis=0)
plt.fill_between(np.arange(len(q1)), q1, q2, color="black", ls="solid", alpha=0.2)

plt.plot(
    np.median(ts_numpy_past[labels_past == 1], axis=0)[:, i], color="red", ls="solid"
)
q1, q2 = np.quantile(ts_numpy_past[labels_past == 1, ..., i], [0.33, 0.66], axis=0)
plt.fill_between(np.arange(len(q1)), q1, q2, color="red", ls="solid", alpha=0.2)

plt.plot(
    np.median(ts_numpy_future[labels_future == 0], axis=0)[:, i],
    color="blue",
    ls="solid",
)
q1, q2 = np.quantile(ts_numpy_future[labels_future == 0, ..., i], [0.33, 0.66], axis=0)
plt.fill_between(np.arange(len(q1)), q1, q2, color="blue", ls="solid", alpha=0.2)

plt.plot(
    np.median(ts_numpy_future[labels_future == 1], axis=0)[:, i],
    color="green",
    ls="solid",
)
q1, q2 = np.quantile(ts_numpy_future[labels_future == 1, ..., i], [0.33, 0.66], axis=0)
plt.fill_between(np.arange(len(q1)), q1, q2, color="green", ls="solid", alpha=0.2)

In [ ]:
i = 4

plt.plot(
    np.median(ts_numpy_past[labels_past == 0], axis=0)[:, i], color="black", ls="solid"
)
q1, q2 = np.quantile(ts_numpy_past[labels_past == 0, ..., i], [0.33, 0.66], axis=0)
plt.fill_between(np.arange(len(q1)), q1, q2, color="black", ls="solid", alpha=0.2)

plt.plot(
    np.median(ts_numpy_past[labels_past == 1], axis=0)[:, i], color="red", ls="solid"
)
q1, q2 = np.quantile(ts_numpy_past[labels_past == 1, ..., i], [0.33, 0.66], axis=0)
plt.fill_between(np.arange(len(q1)), q1, q2, color="red", ls="solid", alpha=0.2)

plt.plot(
    np.median(ts_numpy_future[labels_future == 0], axis=0)[:, i],
    color="blue",
    ls="solid",
)
q1, q2 = np.quantile(ts_numpy_future[labels_future == 0, ..., i], [0.33, 0.66], axis=0)
plt.fill_between(np.arange(len(q1)), q1, q2, color="blue", ls="solid", alpha=0.2)

plt.plot(
    np.median(ts_numpy_future[labels_future == 1], axis=0)[:, i],
    color="green",
    ls="solid",
)
q1, q2 = np.quantile(ts_numpy_future[labels_future == 1, ..., i], [0.33, 0.66], axis=0)
plt.fill_between(np.arange(len(q1)), q1, q2, color="green", ls="solid", alpha=0.2)

In [ ]:
def group_populations(df: pd.DataFrame) -> pd.DataFrame:
    return df.group_by(
        ["member", pl.col("time").dt.year().alias("year")], maintain_order=True
    ).agg(pl.col("labels").value_counts())


labels_split = {
    "past": labels_df.filter(pl.col("time").dt.year().alias("year") < 2025),
    "future": labels_df.filter(pl.col("time").dt.year().alias("year") >= 2025),
}
grouped_pops = {key: group_populations(value) for key, value in labels_split.items()}
labels_first_week_split = {
    key: value.filter(pl.col("time").dt.ordinal_day() < 161)
    for key, value in labels_split.items()
}
grouped_pops_first_week = {
    key: group_populations(value) for key, value in labels_first_week_split.items()
}

In [ ]:
def where_labels(labels, cluster_index):
    member_index, year_index = np.where(labels.reshape(49, 30) == cluster_index)
    return member_index * 30 + year_index


labels_ts_kmeans = {"past": labels_past, "future": labels_future}
index_ts_kmeans = {
    key: {
        cluster_index: where_labels(value, cluster_index)
        for cluster_index in np.unique(value)
    }
    for key, value in labels_ts_kmeans.items()
}

In [ ]:
grouped_pops_first_week["past"]

In [ ]:
def to_pop(grouped, expected_nclu: int = 36):
    pop = (
        grouped.explode("labels")
        .unnest("labels")
        .group_by("labels")
        .agg(pl.col("count").sum())
        .sort("labels")
    )
    pop = (
        pl.Series("labels", np.arange(expected_nclu))
        .to_frame()
        .join(pop, on="labels", how="left")
        .fill_null(0)
    )
    return pop["count"].to_numpy().astype(np.int32)


cluster_pops = {
    period: {
        cluster_index: to_pop(grouped_pops_first_week[period][cluster_indices])
        for cluster_index, cluster_indices in index_dict.items()
    }
    for period, index_dict in index_ts_kmeans.items()
}


def get_rel_pop(grouped, index, expected_nclu: int = 36):
    pop_cluster = to_pop(grouped[index], expected_nclu)
    pop_overall = to_pop(grouped, expected_nclu)
    return pop_cluster / len(index) - pop_overall / grouped.shape[0]


cluster_rel_pops = {
    period: {
        cluster_index: get_rel_pop(grouped_pops_first_week[period], cluster_indices)
        for cluster_index, cluster_indices in index_dict.items()
    }
    for period, index_dict in index_ts_kmeans.items()
}

In [ ]:
# rel

fig, axes = plt.subplots(
    len(cluster_rel_pops),
    len(cluster_rel_pops["past"]),
    figsize=(12, 9),
    subplot_kw={"aspect": "equal"},
)
cmap = colormaps.balance
norm = BoundaryNorm(np.arange(-0.1, 0.12, 0.04), cmap.N)
for axs, (period, dico) in zip(axes, cluster_rel_pops.items()):
    for ax, (cluster, pop) in zip(axs, dico.items()):
        ax.set_title(f"{period}, {cluster}")
        fig, ax = net.plot_on_map(pop, cmap=cmap, norm=norm, fig=fig, ax=ax)

In [ ]:
# abs

fig, axes = plt.subplots(
    len(cluster_rel_pops),
    len(cluster_rel_pops["past"]),
    figsize=(12, 9),
    subplot_kw={"aspect": "equal"},
)
cmap = colormaps.bubblegum_r
norm = BoundaryNorm(np.arange(0, 2100, 400), cmap.N)
for axs, (period, dico) in zip(axes, cluster_pops.items()):
    for ax, (cluster, pop) in zip(axs, dico.items()):
        ax.set_title(f"{period}, {cluster}")
        fig, ax = net.plot_on_map(pop, cmap=cmap, norm=norm, fig=fig, ax=ax)

In [ ]:
pop

### cluster trajectory through SOM space

In [ ]:
labels_np = labels_df["labels"].to_numpy().reshape(49 * 60, 92)
dists = net.neighborhoods.distances.astype(np.uint8)

In [ ]:
dmats = []
for i in range(92):
    dmat = dists[labels_np[:, 0], :][:, labels_np[:, 0]]
    dmats.append(dmat)
dmats = np.stack(dmats, axis=2)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

In [ ]:
aglb = (
    AgglomerativeClustering(
        None, metric="precomputed", linkage="single", distance_threshold=90
    )
    .fit(dmats.sum(axis=2))
    .labels_
)

In [ ]:
np.unique(aglb, return_counts=True)

In [ ]:
(aglb == i).sum()

In [ ]:
labels_df = labels_df.with_columns(year=pl.col("time").dt.year())
grouped = (
    labels_df.filter(pl.col("time").dt.month().is_in([6]))
    .group_by(["member", "year"], maintain_order=True)
    .agg(pl.col("labels").value_counts())
)
# .explode("labels").unnest("labels")
fig, axes = plt.subplots(2, 4, figsize=(16, 6.4), subplot_kw={"aspect": "equal"})
axes = axes.ravel()
j = 0
for i in np.unique(aglb):
    cluster_pop = (aglb == i).sum()
    if cluster_pop < 50:
        continue
    this_pop = (
        grouped.filter(aglb == i)
        .explode("labels")
        .unnest("labels")
        .group_by("labels")
        .agg(pl.col("count").sum() / cluster_pop)
        .sort("labels")
    )
    this_pop = (
        pl.Series("labels", np.arange(36))
        .to_frame()
        .join(this_pop, how="left", on="labels")
        .fill_null(0)
    )
    pop_numpy = this_pop["count"].to_numpy()
    cmap = colormaps.matter
    norm = BoundaryNorm(np.arange(0, 6, 1), cmap.N)
    fig, axes[j] = net.plot_on_map(pop_numpy, fig=fig, ax=axes[j], cmap=cmap, norm=norm)
    axes[j].set_title(f"{i}, {cluster_pop}")
    j = j + 1

In [ ]:
grouped.filter(aglb == i).explode("labels").unnest("labels").group_by("labels").agg(
    pl.col("count").sum()
).sort("labels")